# Step 1 -- Data Collection:

##NOTE: I need to get pictures of houses alongside their data such as:
number of rooms, square footage, prices, maybe year listed too. 

# Images I will take of each home: 
    Take images of front of house, backyard, kitchen, living room, dining room, bathroom. and master bedroom. 
    
I also need some way that the cluade AI can make an inference off of the images provided. (Maybe on modern kitchen or not, is there natural lighting, hard wood floor, does it look luxurious, the hosuing style (modern or buglaow))

Worst case scenario I can not do the iamge analysis part and only do the housing prices prediction. (This could also use house prices from other houses in neightborhoods to get average house price per neightborhhd.)



##Features I could query the LLM to make inferences on the house price:
    Ovreall interior condition, 
    Kitchen Quality (stone countertops, modern applicance), 
    bathroom quality (tile quality, vanity type, shower/tub style),
    natural lighting level (low / medium / high),
    ceiling height, 
    exterior material (brick, siding, stucco, stone)
    garage presence and size (num stalls)
    fencing quality
    lot slope,
    window size 

    house style? (traditional/classic, modern, common residential, luxury, a regional style)

Note:mlater I could also try to factor in neighborhoods as well. Take groups of housing listings from various neightborhoods and analyze their data alongside other factors related to the neighborhood( Walkabilityu, avg income, school ratings, crime index)

In [ ]:
CREATE OR REPLACE STAGE house_data_stage
    DIRECTORY = (ENABLE = TRUE)
    FILE_FORMAT = (TYPE = 'CSV'); 

In [ ]:
CREATE OR REPLACE STAGE house_images 
	DIRECTORY = ( ENABLE = true ) 
	ENCRYPTION = ( TYPE = 'SNOWFLAKE_SSE' );

In [ ]:
CREATE OR REPLACE TABLE house_data (
    house_id INT,
    address STRING, 
    price INTEGER,  
    bedrooms INTEGER, 
    bathrooms INTEGER, 
    sqft INTEGER, 
    url_link STRING,
    image_path STRING
);

In [ ]:
SHOW CURRENT_ACCOUNT()

# Prior to runng the below 2 cells, I added a folder of house images, and a XLSX file to snowflake using SnowCLI

To do so, I used the folowing commands: 
snow stage copy     "C:\Users\<Username>\house_price_predictor\my_custom_dataset\images" "@house_images_stage" --recursive
snow stage copy "C:\Users\1999d\OneDrive\Desktop\house_price_predictor\my_custom_dataset\Housing_Data.csv" "@house_data_stage" 

In [ ]:
LIST @house_data_stage 

In [ ]:
REMOVE @house_images_stage

With the data uploaded to snowflake stages, I can now load it into the table.

NOTE: May need to remove commas from the column titles in the excel file.

In [ ]:
SELECT * 
FROM image_paths_raw;

I have to rerun the below cell each snowflake session since it is a temp table, thus when the session closes I lose all data that existed inside that table. I THINK CHECK LATER

In [ ]:
/*
The below 2 querries load the csv data into a temp table 
that will later be used to create the final table with both csv data and images.
*/
CREATE OR REPLACE TEMP TABLE excel_data_raw (
    house_id INT,
    address STRING,
    price INT,
    bedrooms INT,
    bathrooms INT,
    sqft INT,
    url_link STRING
);

In [ ]:
COPY INTO excel_data_raw
FROM @house_data_stage/Housing_Data.csv
FILE_FORMAT = (
    TYPE = 'CSV'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
);

In [ ]:
SELECT * 
FROM excel_data_raw;

Following loading the csv data into the temp table, I now needed to upload the images the house_data table.

In [ ]:
/*
This table will be used to hold the paths of the images saved in th esnowflake stage.
*/
CREATE OR REPLACE TEMP TABLE image_paths_raw (
    house_id INT,
    image_path STRING UNIQUE
);


In [ ]:
from snowflake.snowpark.functions import col
from snowflake.snowpark.context import get_active_session
import os

# Initialize the snowflake session
sesh = get_active_session()

# List files in the stage
files = sesh.sql("LIST @house_images").collect()

rows = []

for f in files:
    # f.name looks like: '1_1.jpg'
    filename = f.name.replace('house_images/', '')  # remove stage prefix if present

    # Skip directories or weird entries
    if not filename.lower().endswith((".jpg")):
        continue

    # Expect format: houseid_imagenum.png
    if "_" not in filename:
        continue

    # Split into house_id and the rest
    house_id_str, image_num = filename.split("_", 1)

    # Remove extension from house_id part if needed
    house_id = int(house_id_str)

    rows.append((house_id, filename))


# Upload house ID and filenames to SQL table
if rows:
    # Create DataFrame with eahc entry a tuple of house ID and filename 
    df = sesh.create_dataframe(rows, schema=["house_id", "filename"])

    # Save each tuple as a row in the SQL table
    df.write.save_as_table("image_paths_raw", mode="append")
    print(f"Inserted {len(rows)} rows into image_paths_raw.")
else:
    print("No valid image files found — check LIST output.")
    
# print(f"\n\n Rows:\n\n, {rows}")

In [ ]:
SELECT *
FROM image_paths_raw

In [ ]:
/*
Now I can upload the table house_data with the data stored inside of both temp tables above
*/

INSERT INTO house_data (
    house_id,
    address,
    price,
    bedrooms,
    bathrooms,
    sqft,
    url_link,
    image_path
)

SELECT
    i.house_id,
    e.address,
    e.price,
    e.bedrooms,
    e.bathrooms,
    e.sqft,
    e.url_link,
    i.image_path
FROM image_paths_raw i
LEFT JOIN excel_data_raw e
    ON i.house_id = e.house_id;

In [ ]:
SELECT * 
FROM house_data

In [ ]:
LIST @SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_IMAGES;


NOTE: I kept receiving an error while trying to query the LLM as it was saying snowflake cant decrypt or work with client side encryption. So I had to create a new stage in the GUI, and set the stage encryption type used to server side. This was for the below cell.


In [ ]:
# With all of the data uploaded into the SQL talbe, 
# I can now query the LLM to perform an inference on the images.

In [ ]:
SELECT CURRENT_DATABASE(), CURRENT_SCHEMA()

In [ ]:
# Load all image paths from your SQL table
rows = sesh.table("image_paths_raw").collect()

print(rows)

In [ ]:
"""
In this cell I ask a LLM model (Claude-4.5) to make an
inference based on the images it is seeing.
"""

import pandas as pd

from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

# Prompt for architectural style classification
prompt = """
Analyze the provided image of a house. Describe the style of the house by 
choosing the most appropriate term from the following list. 
The response should be a single value.

- Modern
- Ranch-style
- Raised ranch-style
- Two-story traditional
- Bungalow
- Prairie Suburban
- Farmhouse / Modern Farmhouse
- Chalet / Mountain-influenced
- Four Square
- Skinny homes 
- Mediterranean

If you are unable to determine the type from the image, respond with N/A
"""


# Dictionary where:
# key = house_id
# value = list of predictions for that house
house_predictions = {}

for house_id in range(1,21):

    query = f"""
        SELECT AI_COMPLETE(
            'claude-sonnet-4-5',
            '{prompt}',
            TO_FILE('@SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_IMAGES', '{house_id}_1.jpg')
        );
    """

    try:
        prediction = sesh.sql(query).collect()[0][0]
    except Exception as e:
        print(f"Skipping '{house_id}_1.jpg': {e}")
        continue

    # Initialize list if needed
    if house_id not in house_predictions:
        house_predictions[house_id] = []

    house_predictions[house_id].append(prediction)

    print(house_id, prediction)
    
print("\n\n\n")
print(house_predictions)

In [ ]:
# This cell cleans up the values in the 
# house_predictions dictionary created in the above cell.
cleaned = {}

for key, value in house_predictions.items():
    cleaned[key] = value.pop().strip('"')
  
house_predictions = cleaned

print(house_predictions)

# NOTE: Below cell s has images that are not perfectly formatted. I will need to go back and try and fix this. (White bar at top and bottom of images --> is there spme wayto fit the box surrounding the images to the size of the image? Or can I at least change that box background color from white to somehting that blends in better????)

In [ ]:
# This cell displays the images analyzed in gallery format 
# using Streamlit:

# streamlit is used to build the image gallery that the bear 
# images are being displayed in.
import streamlit 
import os
import base64


# Display a title using streamlit
streamlit.title("Houses Being Analyzed:")
streamlit.markdown("### These are only the front exteriors of the homes.")
streamlit.markdown("### The exteriors will be displayed alongside the interiors later.")


cols = streamlit.columns(4)
idx =1

while True:

    try:
        filename = f"{idx}_1.jpg"

        # Load image bytes from stage
        file_bytes = sesh.file.get_stream(f"@HOUSE_IMAGES/{filename}").read()
        encoded = base64.b64encode(file_bytes).decode()
    
        # Choose column (0–3)
        col = cols[(idx - 1) % 4]
    
        # Below uses streamlit to create a grid of 4 columns
        # # where each container in the grid can hold images
        with col:
            streamlit.markdown(
                f"""
                <div style='
                    display: flex;
                    justify-content: center;
                    align-items: center;
                    height: 250px;
                    border: 1px solid grey;
                    padding: 10px;
                    border-radius: 8px;
                    margin-bottom: 10px;
                '>
                    <img src="data:image/jpeg;base64,{encoded}"
                         style="max-width: 100%; max-height: 100%; object-fit: contain;">
                </div>
                <p style='text-align:center; color:gray;'>House {idx}</p>
                """,
                unsafe_allow_html=True
            )

            idx+=1
        
    except Exception:
        print(f"Reached end of houses.")
        break    

In [ ]:
"""
I can also ask LLM to rate the interior of the homes based on 

    Luxury Level (Basic, Mid, High End)
        hardwoord vs laminate, custom cabinetry, high end fixtures, tile quality

    Lot size (Small , medium, large) 
        Tight suburban lot, medium fenced yard, arecage / rural     
"""

In [ ]:
"""
In this cell I ask a LLM model (Claude-3.5) to make an
inference on the luxury level based on the images it is seeing.
"""

import pandas as pd

from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

# Prompt for architectural style classification
prompt = """
Analyze the provided image of a house. Describe the level of luxury of the house by 
choosing the most appropriate term from the following list. 
The response should be a single value.

- Basic:
    Flooring: Laminate, Vinyl plank, basic carpet, repeating patterns 
    Cabinetry: Stock cabinets, standard hardware
    Tile & Bathrooms: Ceramic tile, simple grid layout
    Appliances: Standard white/black/stainless
    Countertops: Laminate, basic quartz
    Windows: Standard vinyl, basic blinds

- Mid:
    Flooring: Engineered hardwood, high‑quality laminate, wider planks
    Cabinetry: Semi‑custom, plywood boxes, soft‑close hinges, upgraded hardware
    Tile & Bathrooms: Porcelain tile, larger formats
    Appliances: Mid‑tier stainless
    Countertops: Mid‑grade quartz, granite
    Windows: Upgraded vinyl

- High End: 
    Flooring: Solid hardwood, exotic woods, herringbone/chevron patterns 
    Cabinetry: Fully custom cabinetry, solid wood
    Tile & Bathrooms: Natural stoneslab walls, 
    Appliances: Luxury brands
    Countertops: High‑end quartz, marble, quartzite, waterfall edges
    Windows: Large architectural windows


If you are unable to determine the luxury level from the image, respond with N/A
"""


# Dictionary where:
# key = house_id
# value = list of predictions for that house
luxury_predictions = {}
house_id = 1 
image_id = 1

while house_id < 22:

    query = f"""
        SELECT AI_COMPLETE(
            'claude-sonnet-4-5',
            '{prompt}',
            TO_FILE('@SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_IMAGES', '{house_id}_{image_id}.jpg')
        );
    """

    try:
        prediction = sesh.sql(query).collect()[0][0]
    except Exception as e:
        print(f"Skipping '{house_id}_{image_id}.jpg': {e}")
        image_id = 1
        house_id +=1
        continue

    prediction_id = str(house_id) + '_' + str(image_id)
    
    # Initialize list if needed
    if prediction_id not in luxury_predictions:
        luxury_predictions[prediction_id] = []

    luxury_predictions[prediction_id].append(prediction)

    image_id += 1

    print(prediction_id, prediction)
    
print("\n\n\n")
print(luxury_predictions)

In [ ]:
"""
In this cell I ask a LLM model (Claude-3.5) to make an
inference on the lot size based on the images it is seeing.
"""

import pandas as pd

from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

# Prompt for architectural style classification
prompt = """
Analyze the provided image of a house. Describe the lot size of the house by 
choosing the most appropriate term from the following list. 
The response should be a single value.

- Small: 
    Tight suburban lot

- Medium: 
    Medium fenced yard
    Wide garage
    full front yard
    driveway fits 2-4 cars
    lot width around 45-70ft

- Large: 
    Arecage / Rural


If you are unable to determine the lot size from the image, respond with N/A
"""


# Dictionary where:
# key = house_id
# value = list of predictions for that house
lotsize_predictions = {}

for house_id in range(1,21):

    query = f"""
        SELECT AI_COMPLETE(
            'claude-sonnet-4-5',
            '{prompt}',
            TO_FILE('@SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_IMAGES', '{house_id}_1.jpg')
        );
    """

    try:
        prediction = sesh.sql(query).collect()[0][0]
    except Exception as e:
        print(f"Skipping '{house_id}_1.jpg': {e}")
        continue

    
    # Initialize list if needed
    if house_id not in lotsize_predictions:
        lotsize_predictions[house_id] = []

    lotsize_predictions[house_id].append(prediction)

    print(house_id, prediction)
    
print("\n\n\n")
print(lotsize_predictions)

In [ ]:
# This cell cleans up the values in the 
# house_predictions dictionary created in the above cell.
cleaned = {}

for key, value in lotsize_predictions.items():
    cleaned[key] = value.pop().strip('"')
  
lotsize_predictions = cleaned

print(lotsize_predictions)

# Section 2: Data Analysis

Here I perform summary statitics, and data visualization to identify patterns anomalies, and relationships between variables.

This is done to give me an understanding of the datas' key characteristics and to prepare it for more complex tasks.


In [ ]:
# This cell will find the most common result for each house in luxury classifications.
# It will give each house a single classification.

luxury_predictions_final = {}

for house_num in range(1, 21):
    house_id = str(house_num)

    # Collect all labels for this house
    labels = []
    for key, value in luxury_predictions.items():
        # key format: "H_I"
        parts = str(key).split("_")
        if len(parts) != 2:
            continue

        if parts[0] == house_id:
            # Handle both string and list values safely
            if isinstance(value, list):
                for v in value:
                    labels.append(str(v))
            else:
                labels.append(str(value))

    # Skip houses with no images
    if not labels:
        continue

    # Count occurrences manually
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1

    # Find the most common label
    most_common_label = None
    highest_count = -1
    for label, count in counts.items():
        if count > highest_count:
            highest_count = count
            most_common_label = label

    luxury_predictions_final[house_id] = most_common_label

print(luxury_predictions_final)

In [ ]:
# This cell cleans the values for the luxury_predictions_final dict:
luxury_predictions = {}

for key, value in luxury_predictions_final.items():
    if isinstance(value, str):
        luxury_predictions[int(key)] = value.strip('"')
    else:
        luxury_predictions[int(key)] = value

print(luxury_predictions)

Below creates new combined sql table containing all prior stored data and inferences made for each house.

In [ ]:
/*
This table will be used to hold the inferences made for type, luxury and lot size for each house.
*/
CREATE OR REPLACE TEMP TABLE house_inferences (
    house_id INT,
    house_type STRING,
    luxury STRING,
    lot_size STRING 
);


In [ ]:
# I now need to add entries to the above temp table. 

from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

for house_id in house_predictions.keys():

    house_type = house_predictions[house_id]
    luxury = luxury_predictions[house_id]
    lot_size = lotsize_predictions[house_id]

    # Clean values (remove extra quotes)
    house_type = str(house_type).strip('"').strip("'")
    luxury = str(luxury).strip('"').strip("'")
    lot_size = str(lot_size).strip('"').strip("'")

    sesh.sql(f"""
        INSERT INTO house_inferences (house_id, house_type, luxury, lot_size)
        VALUES ({house_id}, '{house_type}', '{luxury}', '{lot_size}')
    """).collect()


In [ ]:
SELECT * 
FROM house_inferences;

In [ ]:
from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()



df_data = sesh.table("house_data")
df_inf = sesh.table("house_inferences")

# Remove image_path and deduplicate house_data
df_data_unique = df_data.select(
    "house_id",
    "address",
    "price",
    "bedrooms",
    "bathrooms",
    "sqft",
    "url_link"
).distinct()

# Join on house_id
df_joined = df_data_unique.join(
    df_inf,
    df_data_unique["house_id"] == df_inf["house_id"]
)

# Select final columns (1 row per house)
df_final = df_joined.select(
    df_data_unique["house_id"],
    df_data_unique["address"],
    df_data_unique["price"],
    df_data_unique["bedrooms"],
    df_data_unique["bathrooms"],
    df_data_unique["sqft"],
    df_data_unique["url_link"],
    df_inf["house_type"],
    df_inf["luxury"],
    df_inf["lot_size"]
)

# Save to final table
sesh.sql("DROP TABLE IF EXISTS all_house_data").collect()
df_final.write.save_as_table("all_house_data", mode="append")

In [ ]:
SELECT * 
FROM all_house_data;

In [ ]:
from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

house_data_df = sesh.sql("SELECT * FROM SNOWFLAKE_LEARNING_DB.PUBLIC.ALL_HOUSE_DATA").to_pandas()
#house_data_df

In [ ]:
# Below is used to check the types 
# used in the data frame.
house_data_df.dtypes

In [ ]:
# EXPLORATORY DATA ANALYSIS
# Here I perform summary statitics, and 
# data visualization to identify patterns
# anomalies, and relationships between variables. 
# This is done to give me an understanding of 
# the datas' key characteristics and to prepare 
# it for more complex tasks.

# Step 1: Feature Distribution:

# Step 1: Feature Distribution

# TODO move below cell to further down in project where numeric_df is actually used. 

# NOTE: Ask for each cell below if there are any changeed that need to be made for a price predictoin project:

In [ ]:
# Method 1: A Script Based Approach.
import streamlit 
import altair
import pandas as pd

# Below displays a large header at the top of the page. 
streamlit.header("Feature Distributions")

features_df = house_data_df[["PRICE", "BEDROOMS", "BATHROOMS", "SQFT", "LUXURY", "HOUSE_TYPE", "LOT_SIZE"]]

# Below creates a form named distribution_form
# Forms group widgets together so they only trigger when the user presses a submit button.
with streamlit.form("distribution_form"):
    # Below creates a downdown menu, that contains the 
    # column lable sof the dataframe as options.
    feature = streamlit.selectbox("Select a Feature", features_df.columns)
    
    # Below adds a submit button inside of the form.
    submit_dist = streamlit.form_submit_button("Submit", type = "primary")

# Below chekcs if the form was submitted.
if submit_dist:

    # Below checks if the selected column is numeric. 
    if pd.api.types.is_numeric_dtype(features_df[feature]):
        # If a numeric column was selected, create a histogram
        chart = altair.Chart(features_df).mark_bar().encode(
            # Below creates a chart using the whole DataFrame
            altair.X(f"{feature}:Q", bin = True),
            # tells Altair to draw bars
            y = 'count()',
            # Below colors bars based on the feature value
            # which is converted to nominal (ie. N)
            # scale controls how values map to colors, and below 
            # we use Altair's browns color palette.
            color =  altair.Color(
                    f"{feature}:N",
                    scale = altair.Scale(scheme = "blues"),
                    legend = None
                )
        )
        
    else:
        # If a non-numeric column was selected, create a categorical bar chart
        chart = altair.Chart(features_df).mark_bar().encode(
            x =  altair.X(f"{feature}:N"),
            y = 'count()',
            color = altair.Color(
                    f"{feature}:N",
                    scale = altair.Scale(scheme = "blues"),
                    legend = None
                )
        )
    # Below displays the Altair chart in Streamlit. 
    # Properties sets some styling properties for the chart. 
    # use_Container_width=True makes the chart responsive.
    streamlit.altair_chart(chart.properties(height = 380, title = f"Distribution of {feature}"), use_container_width = True)
     

# STEP 2: FEATURE CORRELATIONS

In [ ]:
# Below displays a title.
streamlit.title("🏠 House Features Explorer")

# Load your DataFrame
# house_data_df = session.table("SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_DATA_DF").to_pandas()

# For now assume df_raw already exists
streamlit.subheader("Dataset Preview") # shows a subheader
streamlit.dataframe(house_data_df.head()) # Creates a datafreame in streamlit

# Correct column type detection
# Below selects the numeric columns from the data frame.
numeric_cols = house_data_df[["PRICE", "BEDROOMS", "BATHROOMS", "SQFT"]].columns.tolist()

# Below selects the non-numeric columns from the data frame.
categorical_cols = house_data_df[["LUXURY", "HOUSE_TYPE", "LOT_SIZE"]].columns.tolist()


streamlit.markdown("### Column Types") # Displays a markdown header
streamlit.write("**Numeric columns:**", numeric_cols) # Writes the numeric columns headings to outout.
streamlit.write("**Categorical columns:**", categorical_cols) # Writes the non-numeric columns headings to output.
 
# -----------------------------
# FEATURE DISTRIBUTION SECTION
# -----------------------------
streamlit.header("📈 Feature Distributions")


# Below creates a form so that the user must click a button to trigger the chart.  
with streamlit.form("feature_form"):
    # Below creates a a select box that contains the data frames column titles. 
    feature = streamlit.selectbox("Select a Feature", features_df.columns)
    #Below creates a submit button.
    submitted = streamlit.form_submit_button("Show Distribution", type = "primary")

# Check if the submit button was pressed.
if submitted:
    # Display a sub header, with the selected feature displayed.
    streamlit.subheader(f"Distribution of `{feature}`")

    # If a numeric Feature was selected → An Altair histogram is displayed
    if feature in numeric_cols:
        chart = (
            altair.Chart(house_data_df)
            .mark_bar()
            .encode(
                altair.X(f"{feature}:Q", bin=altair.Bin(maxbins = 30)),
                y = "count()"
            )
            .properties(height = 350)
        )
        streamlit.altair_chart(chart, use_container_width = True)

    # If a categorical feature was selected → An Altair count plot is displayed.
    elif feature in categorical_cols:
        chart = (
            altair.Chart(house_data_df)
            .mark_bar()

            
            .encode(
                x = altair.X(f"{feature}:N", sort='-y'),
                y = "count()",
                
                color = altair.Color(
                    f"{feature}:N",
                    scale = altair.Scale(scheme="blues"),
                    legend = None
                )

            )
            .properties(height = 350)
        )
        streamlit.altair_chart(chart, use_container_width = True)

    # If neither a numeric or non-numeric feature is selected, a warning message is displayed. 
    else:
        streamlit.warning("This feature type is not supported yet.")

# -----------------------------
# SUMMARY STATISTICS
# -----------------------------
streamlit.header("📊 Summary Statistics")

col1, col2 = streamlit.columns(2) # Creates 2 column containers 

# In column container 1 display the numeric data summary  
with col1:
    streamlit.markdown("### Numeric Summary")
    streamlit.write(house_data_df[numeric_cols].describe())


# In column container 2 display the non-numeric data summary
with col2:
    streamlit.markdown("### Categorical Summary")
    streamlit.write(house_data_df[categorical_cols].describe())

In [ ]:
import numpy  as np
import pandas as pd

def correlation_ratio(categories, values):
    """
    correlation_ratio:
        This function takes in a categorical feature and a a numeric feature and
        it returns a measure of how much of the variance in values is explained 
        by categories.
    """

    # Converts categories into pandas categorical object. 
    categories = pd.Categorical(categories)
    # Extracts the unique category levels. 
    cat_levels = categories.categories
    # Compuates the overall mean of the given numeric category saved in the variable values.
    y_mean = values.mean()

    # Below builds the between-group sum of squares:
    # For each category cat in cat_levels: 
    # values[categories==cat] shows all numeric values belonging in that category. 
    numerator = sum([
        # len(... is the number of observations in that category.
        len(values[categories == cat]) *
        # Below si the squared difference between that category's mean and the overall mean. 
        (values[categories == cat].mean() - y_mean) ** 2
        for cat in cat_levels
    ])
    # Sum above adds the quantity calculated within over all categories.

    # Below is total sum of squares / AKA the total variance in the numeric variable:
    denominator = sum((values - y_mean) ** 2)
    
    # below cpmputes and returns the correlation ratio.
    # If denominator is not equal to 0, compute the proportion of variance explained by categories.
    # If denominator is equal to zero, all values are identical (no variance) and thus we return 0.
    return np.sqrt(numerator / denominator) if denominator != 0 else 0


def mixed_correlation(df):
    """
    mixed_correlation:
        Takes in a pandas dataframe and returns a correlation matrix that 
        works for numeric, categorial, and mixed pairs.
    """
    
    cols = df.columns # Stores columns in the dataframe.
    # Creates an empty square dataframe to store the results. 
    # rows/index as well as the columns of this new dataframe 'corr' are set 
    # to the columns of the 'cols' dataframe. All values are also set to floats.
    corr = pd.DataFrame(index = cols, columns = cols, dtype = float) 

    # Below loops over every pair of columns. 
    # This ensures a full matrix.
    for col1 in cols:
        for col2 in cols:
            # If the 2 columns are the same.
            if col1 == col2:
                # Set the correlation to 1, and skip the rest of the loop.
                corr.loc[col1, col2] = 1.0
                continue

            # Below extracts the 2 columns being compared.
            x = df[col1]
            y = df[col2]

            # Numeric ↔ Numeric
            # If both columns are numeric, we use a Pearson correlation, and store the results in the matrix.
            if pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y):
                corr.loc[col1, col2] = x.corr(y)

            # Categorical ↔ Categorical
            # If both columns are non-numeric, we use a Cramer's V correlation, and store the results in the matrix.
            elif x.dtype == "object" and y.dtype == "object":
                corr.loc[col1, col2] = cramers_v(x, y)

            # Numeric ↔ Categorical
            # If one category is numeric and the other is non-numeric, this checks which is 
            # numeric and orders them correctly for the n_r
            else:
                if pd.api.types.is_numeric_dtype(x):
                    corr.loc[col1, col2] = correlation_ratio(y, x)
                else:
                    corr.loc[col1, col2] = correlation_ratio(x, y)

    return corr # Return the correlation matrix.


In [ ]:
"""
Cramér’s V is a correlation measure for 
categorical variables. 

This measure us used to describe the relationship
between two non-numeric features 
   
Possible return values from a cramers analysis:
    0.0 -- No association at all. 
    0.1-0.3 -- Weak association.
    0.3-0.6 -- Moderate association. 
    0.6-1.0 -- Strong association.

How does it work?
    1) Build a contingency table: 
        Which is like a pivot table 
        counting combinations of categories 

    2) Run a chi-square test on that table 
        Measures how different the observed counts
        are from random chance. 

    3) Normalize the chi-square statistic 
        This allows the result to always be beween 
        0 and 1.

This us a useful meausre since categorical features 
don't have numeric distance, thus you can't compute 
a Pearson correlation. 

"""

from scipy.stats import chi2_contingency


def cramers_v(x, y):
    """
    Calculates cramer's V given the 2 input variables, x and y. This is a 
    measure of how closely related the 2 variables are. 

    x and y are two different columns from the dataframe being processed 
    in the function categorical_correlation. 
    """
    
    confusion_matrix = pd.crosstab(x, y) # Create contingency table.

    # If either variable has only one category → no association
    if confusion_matrix.shape[0] == 1 or confusion_matrix.shape[1] == 1:
        return 0.0

    chi2 = chi2_contingency(confusion_matrix)[0] # runs chi square test and get chi square value.
    n = confusion_matrix.sum().sum() # holds number of observations.
    phi2 = chi2 / n # calculates chi-square value.
    r, k = confusion_matrix.shape # r is number of categroeis in x, and k is number in y.

    # Bias correction
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1)) # correct chi-square for small sample size 
    rcorr = r - ((r - 1) ** 2) / (n - 1) # Corrects effective number of categories
    kcorr = k - ((k - 1) ** 2) / (n - 1) # Corrects effective number of categories

    # Below calculates the denominatorand then checks if it is invlaid. 
    denom = min((kcorr - 1), (rcorr - 1))
    if denom <= 0:
        return 0.0 # If denominator is invalid, return 0.

    return np.sqrt(phi2corr / denom) # Retuens the cramers V value between 0 and 1. 

# Below creates a full correlation matrix for categorical features only.
def categorical_correlation(df):
    """
    Takes a dataframe as input and computes Cramer's V for every pair of columns.
    """
    
    cols = df.columns # Holds a list of the dataframes column names.
    corr = pd.DataFrame(index = cols, columns = cols, dtype = float) # Creates an empty square matrix to fill with Cramers V values.

    # Below loops over every pair of values between column 1 and column 2.
    for col1 in cols:
        for col2 in cols:
            # If column 1 and column 2 are equal (In the diagonal), the feature is said to be perfectly correlated with itself. 
            if col1 == col2:
                corr.loc[col1, col2] = 1.0
            else:
                corr.loc[col1, col2] = cramers_v(df[col1], df[col2])

    return corr # Returns the correlation matrix.


In [ ]:
# Method 1: Script-based approach


corr_matrix = mixed_correlation(features_df) # Gets the correlation matrix of the dataframe.

# 
corr_data = (
    corr_matrix
    .stack() # Converts matrix to series with a multi-index.
    .reset_index(name = 'value') # Turns serries into a dataframe
    # Below renames the dataframes' index coumns 
    .rename(columns = {'level_0': 'feature1', 'level_1': 'feature2'}) 
)


# 
corr_chart = (
    altair.Chart(corr_data) # creates altair chart using the corr_data as the source.

    # Converts each row in corr_data to a rectangle,
    # each of which represent a cell in the heatmap
    .mark_rect() 

    # encode defines how data columns map to visual properties. 
    .encode(
        # Sets feature 1 to the x axis.
        x = altair.X('feature1:N', sort=None),
        # Sets feature 2 to the y axis.
        y = altair.Y('feature2:N', sort=None),
        # sets the color of the correlation value. (uses )
        color = altair.Color('value:Q', scale = altair.Scale(scheme = 'cividis')),
        # Createa a tooltip that displays the naems of both features and its value 
        # when you hover over a cell in the matrix.
        tooltip = [
            alt.Tooltip('feature1:N'),
            alt.Tooltip('feature2:N'),
            alt.Tooltip('value:Q', format = ".3f")
        ]
    )
        
    # Sets the chart width and height and gives it a title..
    .properties(
        width = 500,
        height = 500,
        title = "Mixed-Type Feature Correlation Heatmap"
    )
)

# Displays the Altair chart in the streamlit application.
streamlit.altair_chart(corr_chart, use_container_width = True)



# Results? 
# TODO!!!!!!     

In [ ]:
# Below shows the number of unique data values 
# in each of the columns.
house_data_df.nunique()

# House Type Class Distribution:

This analysis will uncover: The number of unique house types, and the relative positions of each ype in the sample.

In [ ]:
import altair as alt
import streamlit as st

st.header("🏠 House Type Distribution")

st.write("""
Understanding the distribution of house types helps us see:
- Whether the dataset is balanced  
- Which categories dominate  
- How much training data each class contributes  
""")

# Ensure correct column name + clean missing values
df = house_data_df.copy()
df["HOUSE_TYPE"] = df["HOUSE_TYPE"].fillna("Unknown")

# --- BAR CHART ---
bar_chart = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x=alt.X("HOUSE_TYPE:N", sort='-y'),
        y="count()",
        color=alt.Color("HOUSE_TYPE:N", scale=alt.Scale(scheme="cividis")),
        tooltip=["HOUSE_TYPE:N", alt.Tooltip("count()", title="Count")]
    )
    .properties(
        height=350,
        title="Bar Chart: House Type Distribution"
    )
)

# --- PIE CHART ---
pie_chart = (
    alt.Chart(df)
    .mark_arc()
    .encode(
        theta="count()",
        color=alt.Color("HOUSE_TYPE:N", scale=alt.Scale(scheme="cividis")),
        tooltip=["HOUSE_TYPE:N", alt.Tooltip("count()", title="Count")]
    )
    .properties(
        height=350,
        title="Pie Chart: House Type Proportions"
    )
)

col1, col2 = st.columns(2)
col1.altair_chart(bar_chart, use_container_width=True)
col2.altair_chart(pie_chart, use_container_width=True)

# Luxury Level Distribution:
This analysis will uncover: The number of unique luxury levels assigned, and the relative positions of each luxury levels assigned in the sample.

In [ ]:
import altair as alt
import streamlit as st

st.header("House Lot Size Distribution")

st.write("""
Let's examine the distribution of house lot size categories in our dataset. This helps us understand:
- How many samples exist for each lot size category  
- Whether the dataset is balanced or imbalanced  
- The relative proportions of each lot size category in the overall sample  
""")


actual_col = None
for name in house_data_df.columns:
    cleaned = name.strip().upper().replace(" ", "_")
    if cleaned == "LOT_SIZE":
        actual_col = name
        break

if actual_col is None:
    st.error("No lot size column found in the dataset.")
else:
    df = house_data_df.copy()

    # Clean missing values
    df[actual_col] = df[actual_col].fillna("Unknown")

    # --- BAR CHART ---
    bar_chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=f'{actual_col}:N',
            y='count()',
            color=alt.Color(f'{actual_col}:N', scale=alt.Scale(scheme='cividis')),
            tooltip=[f'{actual_col}:N', alt.Tooltip('count()', title='Count')]
        )
        .properties(
            height=350,
            title="Bar Chart: Lot Size Distribution"
        )
    )

    # --- PIE CHART ---
    pie_chart = (
        alt.Chart(df)
        .mark_arc()
        .encode(
            theta='count()',
            color=alt.Color(f'{actual_col}:N', scale=alt.Scale(scheme='cividis')),
            tooltip=[f'{actual_col}:N', alt.Tooltip('count()', title='Count')]
        )
        .properties(
            height=350,
            title="Pie Chart: Lot Size Proportions"
        )
    )

    col1, col2 = st.columns(2)
    col1.altair_chart(bar_chart, use_container_width=True)
    col2.altair_chart(pie_chart, use_container_width=True)

# Lot Size Distribution:

In [ ]:
import altair as alt
import streamlit as st

st.header("House Luxury Level Distribution")

st.write("""
Let's examine the distribution of house luxury levels assigned in our dataset. This helps us understand:
- How many samples exist for each possible level  
- Whether the dataset is balanced or imbalanced  
- The relative proportions of each level in the overall sample  
""")

# Ensure correct column name + clean missing values
df = house_data_df.copy()
df["LUXURY"] = df["LUXURY"].fillna("Unknown")

# --- BAR CHART ---
bar_chart = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x='LUXURY:N',
        y='count()',
        color=alt.Color('LUXURY:N', scale=alt.Scale(scheme='cividis')),
        tooltip=['LUXURY:N', alt.Tooltip('count()', title='Count')]
    )
    .properties(
        height=350,
        title="Bar Chart: Luxury Distribution"
    )
)

# --- PIE CHART ---
pie_chart = (
    alt.Chart(df)
    .mark_arc()
    .encode(
        theta='count()',
        color=alt.Color('LUXURY:N', scale=alt.Scale(scheme='cividis')),
        tooltip=['LUXURY:N', alt.Tooltip('count()', title='Count')]
    )
    .properties(
        height=350,
        title="Pie Chart: Luxury Level Proportions"
    )
)

# Display side-by-side
col1, col2 = st.columns(2)
col1.altair_chart(bar_chart, use_container_width=True)
col2.altair_chart(pie_chart, use_container_width=True)

In [ ]:
import streamlit as st
import altair as alt
import pandas as pd

st.header("📊 Dataset Exploratory Data Analysis")

# -----------------------------------------
# 1. BASIC OVERVIEW
# -----------------------------------------
st.subheader("Dataset Overview")

# Uncomment below if using the real dataset (smaller, but real house sales data)#.  
df = house_data_df.copy()

# Uncomment below if using the randomly generated dataset (Larger, but comp.  
# df = house_data_df2.copy()

# Fix column names (Snowflake → uppercase)
df.columns = [c.upper() for c in df.columns]

# Clean numeric columns
numeric_cols = ["PRICE", "SQFT", "BEDROOMS", "BATHROOMS"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Clean categorical columns
cat_cols = ["HOUSE_TYPE", "LUXURY", "LOT_SIZE"]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

st.write(f"Total Samples: {len(df)}")
st.write(f"Numeric Features: {numeric_cols}")
st.write(f"Categorical Features: {cat_cols}")

st.write(f"Number of House Types: {df['HOUSE_TYPE'].nunique()}")
st.write(f"Number of Luxury Levels: {df['LUXURY'].nunique()}")
st.write(f"Number of Lot Size Categories: {df['LOT_SIZE'].nunique()}")

# -----------------------------------------
# 2. SUMMARY STATISTICS
# -----------------------------------------
st.subheader("Summary Statistics")

st.write(f"Average House Price: ${df['PRICE'].mean():,.0f}")
st.write(f"Average Bedrooms: {df['BEDROOMS'].mean():.2f}")
st.write(f"Average Bathrooms: {df['BATHROOMS'].mean():.2f}")
st.write(f"Average Square Footage: {df['SQFT'].mean():.2f}")

st.write(f"Most Common Lot Size: {df['LOT_SIZE'].mode()[0]}")
st.write(f"Most Common Luxury Level: {df['LUXURY'].mode()[0]}")
st.write(f"Most Common House Type: {df['HOUSE_TYPE'].mode()[0]}")

# -----------------------------------------
# 3. PRICE BINNING
# -----------------------------------------
st.subheader("Price Bins")

df["PRICE_BIN"] = pd.cut(
    df["PRICE"],
    bins=5,
    labels=["Very Low", "Low", "Medium", "High", "Very High"],
    ordered=True
)

price_bin_chart = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x="PRICE_BIN:N",
        y="count()",
        color="PRICE_BIN:N"
    )
    .properties(height=300, width=400, title="Distribution of Price Bins")
)

st.altair_chart(price_bin_chart, use_container_width=True)

# -----------------------------------------
# 4. FEATURE DISTRIBUTIONS BY PRICE BIN
# -----------------------------------------
st.subheader("Feature Distributions by Price Range")

for feature in numeric_cols:
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X(f"{feature}:Q", bin=True),
            y="count()",
            color=alt.Color("PRICE_BIN:N", scale=alt.Scale(scheme="viridis")),
            tooltip=[feature, "PRICE_BIN"]
        )
        .properties(height=300, width=400, title=f"{feature} Distribution by Price Range")
    )
    st.altair_chart(chart, use_container_width=True)

# -----------------------------------------
# 5. CORRELATION HEATMAP
# -----------------------------------------
st.subheader("Correlation Heatmap")

corr = df[numeric_cols].corr()

corr_data = (
    corr.stack()
    .reset_index(name="value")
    .rename(columns={"level_0": "Feature1", "level_1": "Feature2"})
)

corr_chart = (
    alt.Chart(corr_data)
    .mark_rect()
    .encode(
        x="Feature1:N",
        y="Feature2:N",
        color=alt.Color("value:Q", scale=alt.Scale(scheme="cividis")),
        tooltip=["Feature1", "Feature2", "value"]
    )
    .properties(width=400, height=400, title="Correlation Heatmap")
)

st.altair_chart(corr_chart, use_container_width=True)

# -----------------------------------------
# 6. SCATTERPLOTS (PRICE vs each numeric feature)
# -----------------------------------------
st.subheader("Feature Relationships (Scatterplots)")

for feature in numeric_cols:
    if feature != "PRICE":
        scatter = (
            alt.Chart(df)
            .mark_circle(size=60, opacity=0.5)
            .encode(
                x="PRICE:Q",
                y=f"{feature}:Q",
                color=alt.Color("PRICE:Q", scale=alt.Scale(scheme="viridis")),
                tooltip=["PRICE", feature]
            )
            .properties(width=380, height=380, title=f"PRICE vs {feature}")
        )
        st.altair_chart(scatter, use_container_width=True)

# -----------------------------------------
# 7. CATEGORICAL DISTRIBUTIONS
# -----------------------------------------
st.subheader("Categorical Feature Distributions")

for col in cat_cols:
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X(f"{col}:N"),
            y="count()",
            color=alt.Color(f"{col}:N"),
        )
        .properties(width=380, height=380, title=f"Distribution of {col}")
    )

    st.altair_chart(chart, use_container_width=True)


In [ ]:
from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

house_data_df2 = sesh.sql("SELECT * FROM SNOWFLAKE_LEARNING_DB.PUBLIC.ALL_HOUSE_DATA").to_pandas()
#house_data_df2

# Section 3 - Part 1: Data Operations

The below cell is used to generate a radnmoized sample dataset that can be used for ML model training and evaluation. I may move it to the top of the document later, so that the randmoized sample can also be used in section 2 of the project (Data Analysis). 

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

# -----------------------------
# Categories from your dataset
# -----------------------------
house_types = [
    "Modern",
    "Ranch-style",
    "Raised ranch-style",
    "Two-story traditional",
    "Bungalow",
    "Prairie Suburban",
    "Farmhouse / Modern Farmhouse",
    "Chalet / Mountain-influenced",
    "Four Square",
    "Skinny homes",
    "Mediterranean"
]

luxury_levels = ["Basic", "Mid", "High End"]
lot_sizes = ["Small", "Medium", "Large"]

# -----------------------------
# Generate 300 synthetic houses
# -----------------------------
n = 300

house_data_df2 = pd.DataFrame({
    "SQFT": np.random.normal(1800, 450, n).astype(int).clip(900, 4000),
    "BEDROOMS": np.random.choice([2, 3, 4, 5], n, p=[0.15, 0.45, 0.30, 0.10]),
    "BATHROOMS": np.random.choice([1, 2, 3, 4], n, p=[0.10, 0.50, 0.30, 0.10]),
    "HOUSE_TYPE": np.random.choice(house_types, n),
    "LUXURY": np.random.choice(luxury_levels, n, p=[0.5, 0.35, 0.15]),
    "LOT_SIZE": np.random.choice(lot_sizes, n, p=[0.4, 0.45, 0.15])
})

# -----------------------------
# Price model (realistic-ish)
# -----------------------------
base_price = (
    house_data_df2["SQFT"] * np.random.uniform(220, 260) +
    house_data_df2["BEDROOMS"] * 15000 +
    house_data_df2["BATHROOMS"] * 20000
)

# Luxury multipliers
luxury_factor = house_data_df2["LUXURY"].map({
    "Basic": 1.00,
    "Mid": 1.10,
    "High End": 1.25
})

# House type adjustments
type_adjust = house_data_df2["HOUSE_TYPE"].map({
    "Modern": 1.15,
    "Ranch-style": 1.00,
    "Raised ranch-style": 1.05,
    "Two-story traditional": 1.10,
    "Bungalow": 0.95,
    "Prairie Suburban": 1.08,
    "Farmhouse / Modern Farmhouse": 1.12,
    "Chalet / Mountain-influenced": 1.20,
    "Four Square": 1.00,
    "Skinny homes": 0.90,
    "Mediterranean": 1.18
})

# Lot size adjustments
lot_adjust = house_data_df2["LOT_SIZE"].map({
    "Small": 0.95,
    "Medium": 1.00,
    "Large": 1.10
})

# Add noise -- noise of 40000 gives every house (+/-) $40000 of possible 
# randomness

# TODO after I have added the features in the below markdown I will turn noise up to 100000-150000!!! TODO!!!!!!!!!
noise = np.random.normal(0, 40000, n)

house_data_df2["PRICE"] = (base_price * luxury_factor * type_adjust * lot_adjust + noise).astype(int)

house_data_df2.head(10)

The above cell uses a simplified formula to calculate the pric eof each home. I have tried to include more randomness in these equations to make it more realistic. 

This is because houses in real life are priced based off of more than the factors (IE the features) that I am considering here. 

Some of these features include: 
    school rating, 
	crime index, 
	distance to downtown, 
	age of home, renovtion score, 
	walkability, noise level, 
	HOA Fees 
	lot shape, 
	has_garage, 
	garage size,
	park distance,
	year built, 
	renovation score, 
	lot_shape, 
	Market_year, 
	interest_rate, 
	days_on_market, 
	seller_motivation_score, 
	SQFT, 
	Bedrooms, 
	bathrooms, 
	House_type, 
	lot_size, 
	luxury,   

This is ontop of the the other factorts that have been included in my analysis so far.
Also human based motivations such as a desire to set a house a t certain price. Houses are hard to price with a simple formula.

In [ ]:
# Below prepares the features and price:
# Dataframe is separated into 2. Features are assigned to the x variable 
# while the price is assigned to y.
# Prepare features and target

numeric_cols = ["SQFT", "BEDROOMS", "BATHROOMS"]
categorical_cols = ["HOUSE_TYPE", "LUXURY", "LOT_SIZE"]

X = house_data_df2[numeric_cols + categorical_cols]
y = house_data_df2["PRICE"]


# Above sets us up for a regression model.

In [ ]:
# Check for missing data: 

# Data quality checks
# X is the feature matrix, AKA all the columns used to train a model.
# .isnull() creates a dataframe of True/Fal values where True used 
# when value is missing, and false used when value is present.
# Use .sum().sum() as this sums columns wise, giving num missing vals per collumn,
# y is the target variable, AKA the thing we are trying to predict. 
# isnull().sum() counts how many missing values exist in the target column.
missing_features = X.isnull().sum().sum() 
missing_target = y.isnull().sum()

print(f"\n🔍 Data Quality:")
print(f"   Missing feature values: {missing_features}")
print(f"   Missing target values: {missing_target}")

In [ ]:
# Data Splitting:
# Data is separated into training-testing sets using 80/20 ratio in scikit-learn
# Preserves class balance usig stratification. 
# Ensures reproducibility with a fixed random seed
# Prints a clean summary of: number of samples in each split, number of features, and the class distribution.

# Import scikit-learn modules at first use
from sklearn.model_selection import train_test_split

# Below splits data using scikit-learn (recommended by Snowflake for ML operations)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, # X is the feature matrix. y is the target labels.
    test_size = 0.2, # 20% of the data is the test set, thus 80% is the training set.
    random_state = 42, # This ensures the split is reproducible.
    # No stratify for regression
)

print("✅ Data splitting completed!")
print('-' * 35) 

print("📊 Data Split Summary:")
print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Number of features: {X_train.shape[1]}")
print('-' * 35)

print("\n🎯 Target Summary:")
print(f"Training target stats: min={y_train.min():.2f}, max={y_train.max():.2f}, mean={y_train.mean():.2f}")
print(f"Testing target stats: min={y_test.min():.2f}, max={y_test.max():.2f}, mean={y_test.mean():.2f}")
print('-' * 35) 

 # Section 3 - Part 2: Machine Learning Model Training

In [ ]:
# TODO: Ask AI what each line is doing in this cell
# and write a comment explaining it.


# ============================================
# 1. Define numeric + categorical feature lists
# ============================================
numeric_features = ["SQFT", "BEDROOMS", "BATHROOMS"]
categorical_features = ["HOUSE_TYPE", "LOT_SIZE", "LUXURY"]


# ============================================
# 2. Build preprocessing transforme=# ============================================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


# ============================================
# 3. Build pipelines for each model
# ============================================
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR


rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42
    ))
])

knn_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", KNeighborsRegressor(
        n_neighbors=20,
        weights="distance"
    ))
])

xgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror"
    ))
])

lr_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LinearRegression())
])

ridge_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", Ridge(alpha=1.0))
])

lasso_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", Lasso(alpha=0.01, max_iter=5000, tol=1e-3))
])

elastic_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000, tol=1e-3))
])

gbr_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", GradientBoostingRegressor())
])

svr_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", SVR(kernel="rbf", C=100, gamma=0.1))
])

et_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", ExtraTreesRegressor(n_estimators=300, random_state=42))
])

mlp_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),
    max_iter=2000,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.1,
    random_state=42
))
])


# ============================================
# 4. Fit the pipelines
# ============================================
rf_pipeline.fit(X_train, y_train)
knn_pipeline.fit(X_train, y_train)
xgb_pipeline.fit(X_train, y_train)
lr_pipeline.fit(X_train, y_train)
ridge_pipeline.fit(X_train, y_train)
lasso_pipeline.fit(X_train, y_train)
elastic_pipeline.fit(X_train, y_train)
gbr_pipeline.fit(X_train, y_train)
svr_pipeline.fit(X_train, y_train)
et_pipeline.fit(X_train, y_train)
mlp_pipeline.fit(X_train, y_train)


# ============================================
# 5. Save the pipelines
# ============================================
import joblib

joblib.dump(rf_pipeline, "rf_pipeline.pkl")
joblib.dump(knn_pipeline, "knn_pipeline.pkl")
joblib.dump(xgb_pipeline, "xgb_pipeline.pkl")
joblib.dump(lr_pipeline, "lr_pipeline.pkl")
joblib.dump(ridge_pipeline, "ridge_pipeline.pkl")
joblib.dump(lasso_pipeline, "lasso_pipeline.pkl")
joblib.dump(elastic_pipeline, "elastic_pipeline.pkl")
joblib.dump(gbr_pipeline, "gbr_pipeline.pkl")
joblib.dump(svr_pipeline, "svr_pipeline.pkl")
joblib.dump(et_pipeline, "et_pipeline.pkl")
joblib.dump(mlp_pipeline, "mlp_pipeline.pkl")


print("All pipelines trained and saved!")

In [ ]:
# This cell creates a function which gives a summary of the model's performance following training.

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import streamlit as st

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    # Predictions
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Metrics
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

     # Streamlit output
    st.subheader(name)

    results_df = pd.DataFrame({
        "Metric": ["MAE (Train)", "MAE (Test)",
                   "RMSE (Train)", "RMSE (Test)",
                   "R² (Train)", "R² (Test)"],
        "Value": [train_mae, test_mae,
                  train_rmse, test_rmse,
                  train_r2, test_r2]
    })

    st.table(results_df)

    # Add spacing between models
    st.markdown("<br><br>", unsafe_allow_html=True)

# Section 3 - Part 3: Benchmarking of Machine Learning Algorithms

Benchmarking means we are comparing various ML models to see which perform the best and/or are best for the specific use case.

We wish to seelct a model that can generalize well on new, unseen data and one that can provide actionable insights.

We are careful to be aware of the following 2 points:

Model Overfitting: We can evaluate how well the model generalizes on new data by testing the degree the algorithm overfits the data.
Model Interpretability: We ensure our model yields actional insight by understanding the underlying reasons as to 'why' the model made the predictions it does. We can do this by tracing how input features are influencing the output.

In [ ]:
#This cell utilizes the function in the above cell to generate and then display the
# performance of the models.
import streamlit as st


evaluate_model("Random Forest Pipeline", rf_pipeline, X_train, X_test, y_train, y_test)
evaluate_model("KNN Pipeline", knn_pipeline, X_train, X_test, y_train, y_test)
evaluate_model("XGBoost Pipeline", xgb_pipeline, X_train, X_test, y_train, y_test)
evaluate_model("Linear Regression Pipeline", lr_pipeline, X_train, X_test, y_train, y_test)
evaluate_model("Ridge Regression Pipeline", ridge_pipeline, X_train, X_test, y_train, y_test)
evaluate_model("Lasso Regression Pipeline", lasso_pipeline, X_train, X_test, y_train, y_test)
evaluate_model("Elastic Regression Pipeline", elastic_pipeline, X_train, X_test, y_train, y_test)
evaluate_model("Gradient Boosting Regression Pipeline", gbr_pipeline, X_train, X_test, y_train, y_test)
evaluate_model("Support Vector Machine Pipeline", svr_pipeline, X_train, X_test, y_train, y_test)
evaluate_model("ExtraTreesRegressor Pipeline", et_pipeline, X_train, X_test, y_train, y_test)
evaluate_model("MLPRegressor Pipeline", mlp_pipeline, X_train, X_test, y_train, y_test)

# NOTE: After training models like in step 2 of part 3 above cells I should do the below: 

After training your models, you should:
Benchmark all models

Check for overfitting

Visualize prediction errors

✔ Analyze feature importance

✔ Add predictions back into the dataframe

✔ Integrate the best model into Streamlit

✔ (Optional) Perform hyperparameter tuning

✔ (Optional) Use SHAP for interpretability
This is the exact workflow used in professional ML projects.


# Note: In the below cell I display the image of the houses dfront exterior alongside the actual price, and the price predictios made by each of the 3 models. Unfortunately you can see that the price predictions made by each of the models are te exact same. I am unsure as to why this is the case. 

For now I am going to assume it is because of my samepl size. The sample is copmposed of only 20 houses, which is tiny for most ML models

In [ ]:
from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

import streamlit as st
import pandas as pd
from PIL import Image
from io import BytesIO
import base64
import joblib
import numpy as np

# Load trained pipelines (these already include preprocessing)
rf = joblib.load("rf_pipeline.pkl")
knn = joblib.load("knn_pipeline.pkl")
xgb = joblib.load("xgb_pipeline.pkl")
lr = joblib.load("lr_pipeline.pkl")
ridge = joblib.load("ridge_pipeline.pkl")
lasso = joblib.load("lasso_pipeline.pkl")
elastic = joblib.load("elastic_pipeline.pkl")
gbr = joblib.load("gbr_pipeline.pkl")
svr = joblib.load("svr_pipeline.pkl")
et = joblib.load("et_pipeline.pkl")
mlp = joblib.load("mlp_pipeline.pkl")

st.title("🏡 All Houses — Price Predictions")
st.markdown("### Front exteriors with predictions from all 3 models")

# 4-column layout
cols = st.columns(4)

idx = 1

while True:
    try:
        # -----------------------------
        # Load image from Snowflake
        # -----------------------------
        filename = f"{idx}_1.jpg"
        file_bytes = sesh.file.get_stream(f"@HOUSE_IMAGES/{filename}").read()
        img = Image.open(BytesIO(file_bytes))

        # -----------------------------
        # Load matching row of features
        # -----------------------------
        row = house_data_df2.iloc[idx - 1]

        # Actual Price
        actual_price = row["PRICE"]

        # -----------------------------
        # Build DataFrame for prediction
        # Drop PRICE so only features remain
        # -----------------------------
        row_df = pd.DataFrame([row.drop("PRICE")])[X.columns]

        # -----------------------------
        # Predictions (pipeline handles preprocessing)
        # -----------------------------
        pred1 = rf.predict(row_df)[0]
        pred2 = knn.predict(row_df)[0]
        pred3 = xgb.predict(row_df)[0]
        pred4 = lr.predict(row_df)[0]
        pred5 = ridge.predict(row_df)[0]
        pred6 = lasso.predict(row_df)[0]
        pred7 = elastic.predict(row_df)[0]
        pred8 = gbr.predict(row_df)[0]
        pred9 = svr.predict(row_df)[0]
        pred10 = et.predict(row_df)[0]
        pred11 = mlp.predict(row_df)[0]


        # -----------------------------
        # Display in correct column
        # -----------------------------
        col = cols[(idx - 1) % 4]

        with col:
            st.image(img, use_column_width=True)
            st.markdown(f"### House {idx}")
            st.write(f"**Random Forest:** ${pred1:,.2f}")
            st.write(f"**KNN:** ${pred2:,.2f}")
            st.write(f"**XGBoost:** ${pred3:,.2f}")
            st.write(f"**Linear Regression:** ${pred4:,.2f}")
            st.write(f"**Ridge Regression:** ${pred5:,.2f}")
            st.write(f"**Lasso Regression:** ${pred6:,.2f}")
            st.write(f"**Elastic Regression:** ${pred7:,.2f}")
            st.write(f"**GBR:** ${pred8:,.2f}")
            st.write(f"**SVR:** ${pred9:,.2f}") 
            st.write(f"**ET:** ${pred10:,.2f}")
            st.write(f"**MLP:** ${pred11:,.2f}")
            st.write(f"**Actual Price:** ${actual_price:,.2f}")
            st.markdown("---")

        idx += 1

    except Exception as e:
        st.error(f"Error at index {idx}: {e}")
        break


# Benchmarking: How do we becnhmark for regression models?

For regression we can compare:
- MAE (Mean Absolute Error) 
- RMSE (Root Mean Squared Error) 
- R^2 (Coefficient of +Determination) 
- Overfitting (= Train Error -Test Error) 

In [ ]:
# ---------------------------------------------------------
# MODEL BENCHMARKING (SAFE VERSION)
# ---------------------------------------------------------
import streamlit as st

models = [
    ("Random Forest", rf_pipeline),
    ("KNN", knn_pipeline),
    ("XGBoost", xgb_pipeline),
    ("Linear Regression", lr_pipeline),
    ("Ridge", ridge_pipeline),
    ("Lasso", lasso_pipeline),
    ("ElasticNet", elastic_pipeline),
    ("Gradient Boosting", gbr_pipeline),
    ("SVR", svr_pipeline),
    ("ExtraTrees", et_pipeline),
    ("MLPRegressor", mlp_pipeline),
]


import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def benchmark_models(models, X_train, X_test, y_train, y_test):
    results = []

    for name, model in models:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        mae = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2 = r2_score(y_test, preds)

        results.append({
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "R²": r2
        })

    return pd.DataFrame(results)


results_df = benchmark_models(models, X_train, X_test, y_train, y_test)

results_df = results_df.sort_values(by="RMSE")

st.subheader("📊 Model Benchmarking Results")
st.dataframe(results_df.style.format({
    "MAE": "{:,.2f}",
    "RMSE": "{:,.2f}",
    "R²": "{:.4f}"
}))

From the results of the above benchmarking cell, we can see that:

Below is from Microsoft Copilot


- Your linear models are dominating
    Lasso, Linear Regression, Ridge, and ElasticNet are all within a hair of each other.
    This tells you something important:
        The dataset is extremely linear.

    This means:
        The relationship between features and price is mostly additive
            Additive? Model predicts price by adding up weighted contributions from each feature.
        Tree models don't find meaningful nonlinear splits. 
        Neural nets don't find deeper structure 
        Simpler models generalize better 

    This is very common in synthetic or clean housing datasets. 


- Tree models underperform becuase they're overfitting 
    Look at Random Forest: 
        Train RMSE: 44497
        Test RMSE: 57213 

    That'a huge generalization gap.

    Same with XGBoost and ExtraTrees. 
    
    This means:
        Even though you tuned them, the underlying data simply doesn't reward nonlinear models.


- MLPRegressor is doing okay - but still worse than linear models
    MLPRegressor:
        Train RMSE: 32393
        Test RMSE: 41862 

    This is respectable, but still worse than Lasso/Ridge/Linear.

    Why?
        Neural nets need nonlinear structure to beat linear models. 
            The data doesn't have enough of it.


- SVR is basically failing
    SVR Test R^2: 0.0079

    This means:
        It's not scaling well 
        It's extremely sensitive to feature scaling 
        It's not suited to your dataset

    You can safely drop SVR from your model suite.



- The real winner? Lasso!!!
    Lasso edges out the others by a tiny margin since:
        It shrinks coefficients 
        It removes noise 
        It handles multicollinearity well 



In [ ]:
# Model Interpretability: 
# Interpretable ML models are those that provide the variable coefficients that
# directly dictate the relative degree at which it influence the target y values. 

# ---------------------------------------------------------
# FEATURE IMPORTANCE FOR PIPELINE MODELS
# ---------------------------------------------------------

import pandas as pd
import numpy as np

def get_feature_names_from_preprocessor(preprocessor):
    """Extract expanded feature names from ColumnTransformer."""
    output_features = []

    for name, transformer, cols in preprocessor.transformers_:
        if name == "num":
            # numeric features pass through unchanged
            output_features.extend(cols)

        elif name == "cat":
            # categorical features get one-hot encoded
            ohe = transformer
            ohe_features = ohe.get_feature_names_out(cols)
            output_features.extend(ohe_features)

    return output_features


# ---------------------------------------------------------
# Random Forest Feature Importance
# ---------------------------------------------------------

rf_model = rf_pipeline.named_steps["model"]
pre = rf_pipeline.named_steps["preprocess"]

feature_names = get_feature_names_from_preprocessor(pre)

rf_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

st.subheader("🌲 Random Forest — Top 10 Features")
st.dataframe(rf_importance.head(10))


# ---------------------------------------------------------
# XGBoost Feature Importance
# ---------------------------------------------------------

try:
    xgb_model = xgb_pipeline.named_steps["model"]

    xgb_importance = pd.DataFrame({
        "feature": feature_names,
        "importance": xgb_model.feature_importances_
    }).sort_values("importance", ascending=False)

    st.subheader("🚀 XGBoost — Top 10 Features")
    st.dataframe(xgb_importance.head(10))

except Exception as e:
    st.warning(f"XGBoost importance unavailable: {e}")

In [ ]:
# Model Interpretability: 
# Interpretable ML models are those that provide the variable coefficients that
# directly dictate the relative degree at which it influence the target y values. 


# This cell calculates the top 10 Feature Importance:
import pandas as pd
import numpy as np
import streamlit as st


# ---------------------------------------------------------
# Helper: Extract expanded feature names from ColumnTransformer
# ---------------------------------------------------------
def get_feature_names(preprocessor):
    feature_names = []

    for name, transformer, cols in preprocessor.transformers_:
        if name == "num":
            feature_names.extend(cols)

        elif name == "cat":
            ohe = transformer
            ohe_features = ohe.get_feature_names_out(cols)
            feature_names.extend(ohe_features)

    return feature_names


# ---------------------------------------------------------
# Helper: Display top 10 importances
# ---------------------------------------------------------
def show_top_features(model_name, feature_names, importances):
    df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)

    st.subheader(f"🔍 Top 10 Features — {model_name}")
    st.dataframe(df.head(10))


# ---------------------------------------------------------
# Helper: Display top 10 coefficients (linear models)
# ---------------------------------------------------------
def show_top_coefficients(model_name, feature_names, coefs):
    df = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefs
    }).sort_values("coefficient", key=lambda x: abs(x), ascending=False)

    st.subheader(f"📐 Top 10 Coefficients — {model_name}")
    st.dataframe(df.head(10))


# ---------------------------------------------------------
# MODELS + PIPELINES
# ---------------------------------------------------------
models = [
    ("Random Forest", rf_pipeline),
    ("ExtraTrees", et_pipeline),
    ("Gradient Boosting", gbr_pipeline),
    ("XGBoost", xgb_pipeline),
    ("Linear Regression", lr_pipeline),
    ("Ridge", ridge_pipeline),
    ("Lasso", lasso_pipeline),
    ("ElasticNet", elastic_pipeline),
]


# ---------------------------------------------------------
# MAIN LOOP
# ---------------------------------------------------------
st.title("📊 Model Interpretability — Top 10 Features")

for name, pipeline in models:

    pre = pipeline.named_steps["preprocess"]
    model = pipeline.named_steps["model"]

    feature_names = get_feature_names(pre)

    # Tree-based models → feature_importances_
    if hasattr(model, "feature_importances_"):
        show_top_features(name, feature_names, model.feature_importances_)

    # Linear models → coefficients
    elif hasattr(model, "coef_"):
        show_top_coefficients(name, feature_names, model.coef_)

    else:
        st.warning(f"{name} does not provide feature importances or coefficients.")


The above cell calculates the top 10 feature importance for Random Forest, ExtraTrees, Gradient Boosting, and XGBoost. 

This is because the following models do not have feature importances:
	Linear regression, 
	Ridge, 
	Lasso, 
	ElasticNet, 
	KNN, 
	SVR, 
	MLPRegressor.

But linear models do have coefficients, which we also extract. 

So your final cell will include:
	Tree models --> feature_importances_

	Linear Models --> Coefficients 

	MLP / KNN / SVR --> No interpretability (we skip them cleanly)


What is SHAP interpretability?
	An approach to explain the output of any machine learning model by measuring the contribution of each feature to individual predictions

The below SHAP interpretability cell is used for the following models:
	Tree Models:
		Random Forest 
		ExtraTrees 
		Gradient Boosting 
		NOTE: I couldn't get it to work with XGBoost unfortunatley so I removed it from the below cell. 

	LinearExplainer (Fast):
		Linear Regression
		Ridge 
		Lasso	
		ElasticNet

In [ ]:
# ===============================================
# SHAP INTERPRETABILITY FOR ALL SUPPORTED MODELS
# ===============================================

import shap
import pandas as pd
import numpy as np
import streamlit as st
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------
# Helper: Extract expanded feature names from ColumnTransformer
# ---------------------------------------------------------
def get_feature_names(preprocessor):
    feature_names = []
    for name, transformer, cols in preprocessor.transformers_:
        if name == "num":
            feature_names.extend(cols)
        elif name == "cat":
            ohe = transformer
            ohe_features = ohe.get_feature_names_out(cols)
            feature_names.extend(ohe_features)
    return feature_namesr5tttttt4


# ---------------------------------------------------------
# Models that support SHAP (XGBoost REMOVED)
# ---------------------------------------------------------
shap_models = [
    ("Random Forest", rf_pipeline),
    ("ExtraTrees", et_pipeline),
    ("Gradient Boosting", gbr_pipeline),
    ("Linear Regression", lr_pipeline),
    ("Ridge", ridge_pipeline),
    ("Lasso", lasso_pipeline),
    ("ElasticNet", elastic_pipeline),
]


# ---------------------------------------------------------
# MAIN LOOP
# ---------------------------------------------------------
st.title("🔍 SHAP Interpretability for All Models")

for name, pipeline in shap_models:

    st.header(f"📊 SHAP Analysis — {name}")

    pre = pipeline.named_steps["preprocess"]
    model = pipeline.named_steps["model"]

    # Extract feature names
    feature_names = get_feature_names(pre)

    # Transform training data
    X_train_transformed = pre.transform(X_train)
    X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names)

    # -----------------------------------------------------
    # TREE MODELS → TreeExplainer
    # -----------------------------------------------------
    if hasattr(model, "feature_importances_"):
        st.write("Using **TreeExplainer**")

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_train_df)

        # Summary bar plot
        st.subheader("🌎 Global Feature Importance (Bar)")
        fig, ax = plt.subplots(figsize=(10, 6))
        shap.summary_plot(shap_values, X_train_df, plot_type="bar", show=False)
        st.pyplot(fig)

        # Beeswarm plot
        st.subheader("🐝 SHAP Beeswarm Plot")
        fig, ax = plt.subplots(figsize=(10, 6))
        shap.summary_plot(shap_values, X_train_df, show=False)
        st.pyplot(fig)

    # -----------------------------------------------------
    # LINEAR MODELS → LinearExplainer
    # -----------------------------------------------------
    elif hasattr(model, "coef_"):
        st.write("Using **LinearExplainer**")

        explainer = shap.LinearExplainer(model, X_train_df)
        shap_values = explainer.shap_values(X_train_df)

        # Summary bar plot
        st.subheader("🌎 Global Feature Importance (Bar)")
        fig, ax = plt.subplots(figsize=(10, 6))
        shap.summary_plot(shap_values, X_train_df, plot_type="bar", show=False)
        st.pyplot(fig)

        # Beeswarm plot
        st.subheader("🐝 SHAP Beeswarm Plot")
        fig, ax = plt.subplots(figsize=(10, 6))
        shap.summary_plot(shap_values, X_train_df, show=False)
        st.pyplot(fig)

    else:
        st

# Note on SHAP values:
- Negative SHAP values --> Decrease the models prediction 
    - Small SQFT pushes price down
- Positive SHAP values increased the prediction 
    - High SQFT pushes price up


- SHAP values are additive: 
    prediction by model == Baseline + Sum(SHAP values)

In [ ]:
# ============================================
# Top Feature Importance Visualization (Random Forest)
# ============================================

import altair as alt
import pandas as pd

# Extract preprocessor + model from pipeline
pre = rf_pipeline.named_steps["preprocess"]
rf_model = rf_pipeline.named_steps["model"]

# Get expanded feature names
def get_feature_names(preprocessor):
    feature_names = []
    for name, transformer, cols in preprocessor.transformers_:
        if name == "num":
            feature_names.extend(cols)
        elif name == "cat":
            ohe = transformer
            ohe_features = ohe.get_feature_names_out(cols)
            feature_names.extend(ohe_features)
    return feature_names

feature_names = get_feature_names(pre)

# Build importance DataFrame
rf_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

# Select top N
top_n = 5
chart_data = rf_importance.head(top_n)

# Altair bar chart
chart = (
    alt.Chart(chart_data)
    .mark_bar()
    .encode(
        x=alt.X("importance:Q", title="Importance"),
        y=alt.Y("feature:N", title="Feature", sort="-x"),
        color=alt.value("#00B8E7"),
        tooltip=[
            alt.Tooltip("feature:N", title="Feature"),
            alt.Tooltip("importance:Q", title="Importance", format=".4f"),
        ],
    )
    .configure(background="transparent")
    .configure_axis(labelColor="white", titleColor="white")
    .configure_title(color="white")
    .properties(
        title=f"Top {top_n} Feature Importances (Random Forest Regressor)",
        width=600,
        height=400,
    )
)

chart

In [ ]:
# ============================================
# Interpreting Random Forest Models (Regression)
# ============================================

# Build DataFrame of feature importances
rf_feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

# Print top 5
print("✨ Top 5 Most Important Features (Random Forest Regressor):")
print(rf_feature_importance.head(5))

# TODO: the below feature importance cell for XGBoost is bugged and wont work --> Try to fix later!!!!!! 


Once you do fix it, ensure you lace a section in the SHAP cell for XGBoost and change the comment in the markup cell above the SHAP cell to say that the SHAP cell now supports XGBoost.

In [ ]:
# ============================================
# Top Feature Importance Visualization (XGBoost Regressor)
# ============================================

import altair as alt
import pandas as pd

# Extract preprocessor + model from pipeline
pre = xgb_pipeline.named_steps["preprocess"]
xgb_model = xgb_pipeline.named_steps["model"]

# Extract expanded feature names
def get_feature_names(preprocessor):
    feature_names = []
    for name, transformer, cols in preprocessor.transformers_:
        if name == "num":
            feature_names.extend(cols)
        elif name == "cat":
            ohe = transformer
            ohe_features = ohe.get_feature_names_out(cols)
            feature_names.extend(ohe_features)
    return feature_names

feature_names = get_feature_names(pre)

# Build importance DataFrame
xgb_feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

# Select top N
top_n = 5
chart_data = xgb_feature_importance.head(top_n)

# Altair bar chart
chart = (
    alt.Chart(chart_data)
    .mark_bar()
    .encode(
        x=alt.X("importance:Q", title="Importance"),
        y=alt.Y("feature:N", title="Feature", sort="-x"),
        color=alt.value("#FFB347"),  # Orange for XGBoost
        tooltip=[
            alt.Tooltip("feature:N", title="Feature"),
            alt.Tooltip("importance:Q", title="Importance", format=".4f"),
        ],
    )
    .configure(background="transparent")
    .configure_axis(labelColor="white", titleColor="white")
    .configure_title(color="white")
    .properties(
        title=f"Top {top_n} Feature Importances (XGBoost Regressor)",
        width=600,
        height=400,
    )
)

chart

In [ ]:
# This cell computes permutation importance for KNN Regressor

""" 
This gives us a ranked list oof features that matter ost for KNN 
 
While KNN has no internal importance mechanism, permutation important revelas:
- Which features KNN relies on most
- How much each feature affect prediction error
- Which featutes are irrelvant

The cell uses MAE as the scoring metric -- Whcih makes the importance values directly tied to price preidction 

"""  


# ============================================
# Permutation Importance for KNN (Pipeline Version)
# ============================================

from sklearn.inspection import permutation_importance
import pandas as pd
import streamlit as st

st.subheader("🔍 Permutation Importance — KNN Regressor")

# Extract preprocessor + model from pipeline
pre = knn_pipeline.named_steps["preprocess"]
knn_model = knn_pipeline.named_steps["model"]

# Extract expanded feature names
def get_feature_names(preprocessor):
    feature_names = []
    for name, transformer, cols in preprocessor.transformers_:
        if name == "num":
            feature_names.extend(cols)
        elif name == "cat":
            ohe = transformer
            ohe_features = ohe.get_feature_names_out(cols)
            feature_names.extend(ohe_features)
    return feature_names

feature_names = get_feature_names(pre)

# Transform X_test using the pipeline's preprocessor
X_test_transformed = pre.transform(X_test)

# Compute permutation importance
knn_perm = permutation_importance(
    knn_model,
    X_test_transformed,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

# Build DataFrame
knn_feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": knn_perm.importances_mean,
    "std": knn_perm.importances_std
}).sort_values("importance", ascending=False)

st.write("✨ Top 5 Most Important Features (KNN Permutation Importance):")
st.dataframe(knn_feature_importance.head(5))

In [ ]:
# ============================================
# KNN Permutation Importance Visualization
# ============================================

import altair as alt

top_n = 5
chart_data = knn_feature_importance.head(top_n)

chart = (
    alt.Chart(chart_data)
    .mark_bar()
    .encode(
        x=alt.X("importance:Q", title="Importance (Increase in MAE)"),
        y=alt.Y("feature:N", title="Feature", sort="-x"),
        color=alt.value("#9B59B6"),
        tooltip=[
            alt.Tooltip("feature:N", title="Feature"),
            alt.Tooltip("importance:Q", title="Importance", format=".4f"),
            alt.Tooltip("std:Q", title="Std Dev", format=".4f"),
        ],
    )
    .configure(background="transparent")
    .configure_axis(labelColor="white", titleColor="white")
    .configure_title(color="white")
    .properties(
        title=f"Top {top_n} Feature Importances (KNN Permutation Importance)",
        width=600,
        height=400,
    )
)

chart

# Section 4 - Experiment Tracking:

# Section 4 - Part 1: Data Loading

In [ ]:
# This cell ignores the 3 categories of warnings shown below: 

import warnings

# Filter out ResourceWarnings. These typically occur when libraries open temporary files 
# or network connections, and they are being ignored since they are harmless in ML notebooks. 
warnings.filterwarnings('ignore', category = ResourceWarning)

# Filter out DeprecationWarning. These typically tell you a function or parameter will be removed in 
# a future version. These are ignored as they are unneeded information during analysis reports.
warnings.filterwarnings('ignore', category = DeprecationWarning)

# Filter out UserWarning are general warnings triggered by libraries when something might be unexpected but not harmful. 
warnings.filterwarnings('ignore', category = UserWarning)

print("⚙️ Warning filters applied — experiment logs will be cleaner.")

In [ ]:
# This cell loads the dataset from the combined sql table and displays it 
# with streamlit.

from snowflake.snowpark.context import get_active_session

# Initialize the snowflake session
sesh = get_active_session()

# Sesh.table uses the snowflake session to convert the table named HOUSE_DATA_FINAL into a 
# pandas dataframe.
# We use the beloe line if we wish to use the actual house dat dataset.
# house_data_df3 = sesh.table('SNOWFLAKE_LEARNING_DB.PUBLIC.ALL_HOUSE_DATA').to_pandas()

# We use the below line if we wih to use the artificial dataset.
house_data_df3 = house_data_df2

# Below prints a title and the dimensions of the dataframe.
streamlit.write("📊 House Dataset Loaded:")
#streamlit.write(f"Shape: {house_data_df3.shape}")

#house_data_df3 # Displays the dataframe.

# Section 4 - Part 2: Data Preprocessing

In [ ]:
house_data_df3.columns # Gives a Pandas Index object of the names of all the columns in the dataframe.

In [ ]:
# ============================================
# Section 4 — Experiment Tracking
# Part 2 — Data Preprocessing (Features + Target)
# ============================================

from sklearn.model_selection import train_test_split
import streamlit as st

# --------------------------------------------
# 1. Clean PRICE column (fix bracketed strings)
# --------------------------------------------
def clean_price_column(df):
    df["PRICE"] = df["PRICE"].apply(
        lambda x: float(str(x).replace("[", "").replace("]", ""))
    )
    return df

house_data_df3 = clean_price_column(house_data_df3)

# --------------------------------------------
# 2. Define feature matrix X and target y
# --------------------------------------------
X = house_data_df3.drop(columns=["PRICE"])
y = house_data_df3["PRICE"]

st.write("🧹 **Data Preprocessing Complete**")
st.write(f"Feature matrix shape: {X.shape}")
st.write(f"Target vector shape: {y.shape}")

In [ ]:
# ============================================
# Section 4 — Experiment Tracking
# Part 2 — Data Quality Check
# ============================================

import streamlit as st

# Count missing values in features and target
missing_features = X.isnull().sum().sum()
missing_target = y.isnull().sum()

# Display results
st.subheader("🔍 Data Quality Check")
st.write(f"Missing feature values: `{missing_features}`")
st.write(f"Missing target values: `{missing_target}`")

In [ ]:
# Below splits the dataset into training and testing sets:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, # x is the features and y is the target labels 
    test_size = 0.2, # Sets the test set to be 20% of the data 
    random_state = 42, # Random_state is a seed for the random number generator.
    # No stratify for regression
)

st.write("✅ Data preparation completed!")

# Below displays the spliut summary, and this shows:
# Number of training samples, number of testing samples, percentage split, number of features (Columns)
st.subheader("📊 Data Split Summary:")
st.write(f"Training set: `{X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)`")
st.write(f"Testing set: `{X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)`")
st.write(f"Number of features: {X_train.shape[1]}")

# Below shows the class distribution in each split:
# This is done to ensure stratification worked, confirms no class was lost during splitting,
# helps detect imbalance issues early. 
st.subheader("🎯 Target Summary:")
st.write(f"Training target stats: `{y_train.describe().to_dict()}`")
st.write(f"Testing target stats: `{y_test.describe().to_dict()}`")

In [ ]:
# We perform the following preprocessing: 
# 1) StandardScaler for numerical features. This transforms features to have mean=0 
# and std=1 
# 2) OneHotEncoder for categorical features (luxury lelve, house type, lot size)
# This converts categorical variables into binary columns. 
# ============================================
# Section 4 — Experiment Tracking
# Part 4 — Preprocessing Pipeline
# ============================================

import streamlit as st
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify numerical and categorical columns
numerical_features = ["SQFT", "BEDROOMS", "BATHROOMS"]
categorical_features = ["HOUSE_TYPE", "LUXURY", "LOT_SIZE"]

# Preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

st.write("🔧 **Preprocessing pipeline created successfully!**")

st.subheader("📌 Feature Groups")
st.write(f"Numerical features: `{numerical_features}`")
st.write(f"Categorical features: `{categorical_features}`")

In [ ]:
# Save scaled test data + actual price values to Snowflake

import streamlit as st

# Create a copy of the raw test features
test_df = X_test.copy()

# Add the actual price values (regression target)
test_df['ACTUAL_PRICE'] = y_test.values

# Convert pandas DataFrame to Snowpark DataFrame
snowpark_df = sesh.create_dataframe(test_df)

# Write to Snowflake table
snowpark_df.write.mode("overwrite").save_as_table(
    "SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_TEST_DATA"
)

# Display results
streamlit.write("✅ Test data successfully saved to HOUSE_TEST_DATA table!")
streamlit.write(f"Number of rows: {len(test_df)}")
streamlit.write(f"Number of columns: {len(test_df.columns)}")

In [ ]:
# Outlier Detection & Removal using IQR

import streamlit as st

st.subheader("🧹 Outlier Detection & Removal (IQR Method)")

# Compute IQR on PRICE
Q1 = house_data_df3["PRICE"].quantile(0.25)
Q3 = house_data_df3["PRICE"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

st.write(f"Lower bound: `{lower_bound:.2f}`")
st.write(f"Upper bound: `{upper_bound:.2f}`")

# Remove outliers BEFORE splitting
before_rows = len(house_data_df3)

house_data_df3 = house_data_df3[
    (house_data_df3["PRICE"] >= lower_bound) &
    (house_data_df3["PRICE"] <= upper_bound)
]

after_rows = len(house_data_df3)

st.write(f"Removed `{before_rows - after_rows}` outlier rows.")
st.write(f"Remaining rows: `{after_rows}`")

In [ ]:
# Rare categoriy handling: 

# If a categorical feature has categories 
# that appear only 1–2 times, one‑hot encoding will create:
# Unstable columns, noise and overfitting risk. 
# So by grouping these rare categories into other, we can prevent his.

# Rare Category Handling for Categorical Features

import streamlit as st

st.subheader("🔎 Rare Category Handling")

threshold = 10

# Explicitly list categorical columns (your dataset uses uppercase)
categorical_cols = ["HOUSE_TYPE", "LUXURY", "LOT_SIZE"]

for col in categorical_cols:
    value_counts = house_data_df3[col].value_counts()
    rare_categories = value_counts[value_counts < threshold].index

    if len(rare_categories) > 0:
        house_data_df3.loc[:, col] = house_data_df3[col].replace(rare_categories, "Other")
        st.write(f"Column `{col}`: grouped {len(rare_categories)} rare categories into 'Other'")

In [ ]:
# Feature engineering: 

import streamlit as st

st.subheader("🛠️ Feature Engineering")

new_features = []

# 1. Price per square foot
house_data_df3.loc[:, "PRICE_PER_SQFT"] = (
    house_data_df3["PRICE"] / house_data_df3["SQFT"]
)
new_features.append("PRICE_PER_SQFT")

# 2. Bedrooms per square foot
house_data_df3.loc[:, "BEDROOMS_PER_SQFT"] = (
    house_data_df3["BEDROOMS"] / house_data_df3["SQFT"]
)
new_features.append("BEDROOMS_PER_SQFT")

# 3. Bathroom-to-bedroom ratio
house_data_df3.loc[:, "BATH_BED_RATIO"] = (
    house_data_df3["BATHROOMS"] / house_data_df3["BEDROOMS"]
)
new_features.append("BATH_BED_RATIO")

st.write("New engineered features added:")
st.write(new_features)

Other possible engineered feature we may want to use in a home price prediction project:

🏠 Structural Features
- total_rooms = bedrooms + bathrooms
- rooms_per_sqft = total_rooms / sqft
- bathroom_to_bedroom_ratio = bathrooms / bedrooms
- finished_sqft_ratio = finished_sqft / total_sqft

📍 Location Features
If you have coordinates or neighbourhood:
- is_urban = 1 if neighbourhood in urban_list else 0
- distance_to_city_center
- school_quality_score
- crime_rate_index

🕒 Time‑Based Features
If you have year built or renovation year:
- house_age = current_year - year_built
- years_since_renovation = current_year - renovation_year

💰 Price‑Related Features
- price_per_sqft = price / sqft
- lot_price_ratio = price / lot_size
- luxury_flag = 1 if price > threshold else 0

🌳 Lot / Exterior Features
- lot_size_per_sqft = lot_size / sqft
- garage_space_ratio = garage_spaces / bedrooms

🧹 Binary Flags
- has_garage
- has_basement
- has_fireplace
- is_renovated

TODO: have the ML models perform an analysis on the data using the engineered features??? BUT before you do so, try to create a moe exrtensive list of engineered features using the above markdown cell as reference.

# Section 4 - Part 3: Experiment Tracking Setup

In [ ]:
# We do experiment tracking using snowflakes ML experiemnt tracking 
# This allows us to log and compare different models and their hyperparameters.

# Below imports snowflake's experiment tracking libary. This lets us log models, metrics 
# hyperparameters, compare runs, store artifacts, and keep a history of experiments.
from snowflake.ml.experiment.experiment_tracking import ExperimentTracking

# Create an ExperimentTracking object that is tracking the current snowflake session.
exp = ExperimentTracking(session = sesh)

# Below sets the experiment name
experiment_name = "House_Price_Modeling_Experiment"
# Below ensures snowflake logs all future model runs. 
exp.set_experiment(experiment_name)

# Below confirms the ExperimentTracking is initialized.
st.write(f"✅ Experiment Tracking Initialized: `{experiment_name}`")

In [ ]:
# This cell does the following:
# Trains the random forest model, evaluates the model, logs hyperparameters to snowflake,
# logs metrics, and stores the results inside of the ExperimentTracker

import streamlit as st
import numpy as np
from datetime import datetime

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --------------------------------------------
# 1. Collect all pipelines into a dictionary
# --------------------------------------------
models = {
    "RandomForestRegressor": rf_pipeline,
    "KNNRegressor": knn_pipeline,
    "XGBRegressor": xgb_pipeline,
    "LinearRegression": lr_pipeline,
    "Ridge": ridge_pipeline,
    "Lasso": lasso_pipeline,
    "ElasticNet": elastic_pipeline,
    "GradientBoostingRegressor": gbr_pipeline,
    "SVR": svr_pipeline,
    "ExtraTreesRegressor": et_pipeline,
    "MLPRegressor": mlp_pipeline
}

# --------------------------------------------
# 2. Reusable experiment‑tracking function
# --------------------------------------------
def run_experiment(model_name, pipeline):
    run_name = f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    with exp.start_run(run_name):

        # Extract and log hyperparameters
        model_params = pipeline.named_steps["model"].get_params()
        exp.log_params(model_params)

        # Train model
        pipeline.fit(X_train, y_train)

        # Predict
        y_pred = pipeline.predict(X_test)

        # Compute metrics
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        # Log metrics
        exp.log_metric("MAE", mae)
        exp.log_metric("RMSE", rmse)
        exp.log_metric("R2", r2)

        # Streamlit output
        st.subheader(f"📊 {model_name} Performance")
        st.write(f"- MAE: `{mae:.4f}`")
        st.write(f"- RMSE: `{rmse:.4f}`")
        st.write(f"- R² Score: `{r2:.4f}`")


# --------------------------------------------
# 3. Run all models
# --------------------------------------------
st.write("🚀 **Running all models and logging results to Snowflake...**")

for model_name, pipeline in models.items():
    run_experiment(model_name, pipeline)

st.success("✅ All models trained, evaluated, and logged successfully!")

# Section 4 - Part 4: Hyperparameter Tuning:

In [ ]:
# Now we are going to analyze the results from our 
# random Forest hyperparameter tuning epxeriments. 

# Below imports all that is needed:
import streamlit as st
import numpy as np
from datetime import datetime
import itertools

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

st.subheader("🔧 Hyperparameter Tuning for All Models")

# ---------------------------------------------------------
# 1. Define hyperparameter grids for each model
# ---------------------------------------------------------
param_grids = {
    "RandomForestRegressor": {
        "n_estimators": [100, 200],
        "max_depth": [5, 10],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"]
    },
    "KNNRegressor": {
        "n_neighbors": [5, 10, 20],
        "weights": ["uniform", "distance"]
    },
    "XGBRegressor": {
        "n_estimators": [300, 500],
        "learning_rate": [0.05, 0.1],
        "max_depth": [4, 6]
    },
    "LinearRegression": {
        # No hyperparameters to tune, but we include a dummy grid
        "fit_intercept": [True]
    },
    "Ridge": {
        "alpha": [0.1, 1.0, 10.0]
    },
    "Lasso": {
        "alpha": [0.001, 0.01, 0.1]
    },
    "ElasticNet": {
        "alpha": [0.001, 0.01],
        "l1_ratio": [0.3, 0.5, 0.7]
    },
    "GradientBoostingRegressor": {
        "n_estimators": [200, 300],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 4]
    },
    "SVR": {
        "C": [10, 100],
        "gamma": ["scale", 0.1]
    },
    "ExtraTreesRegressor": {
        "n_estimators": [200, 300],
        "max_depth": [None, 10]
    },
    "MLPRegressor": {
        "hidden_layer_sizes": [(128, 64), (256, 128, 64)],
        "learning_rate_init": [0.001, 0.01]
    }
}

# ---------------------------------------------------------
# 2. Model dictionary (pipelines already defined earlier)
# ---------------------------------------------------------
models = {
    "RandomForestRegressor": rf_pipeline,
    "KNNRegressor": knn_pipeline,
    "XGBRegressor": xgb_pipeline,
    "LinearRegression": lr_pipeline,
    "Ridge": ridge_pipeline,
    "Lasso": lasso_pipeline,
    "ElasticNet": elastic_pipeline,
    "GradientBoostingRegressor": gbr_pipeline,
    "SVR": svr_pipeline,
    "ExtraTreesRegressor": et_pipeline,
    "MLPRegressor": mlp_pipeline
}

# ---------------------------------------------------------
# 3. Storage for leaderboard
# ---------------------------------------------------------
tuning_results = []

# ---------------------------------------------------------
# 4. Hyperparameter tuning loop (ALL MODELS)
# ---------------------------------------------------------
for model_name, pipeline in models.items():

    st.write(f"### 🔍 Tuning {model_name}")

    grid = param_grids[model_name]

    # Generate all combinations
    param_combinations = [
        dict(zip(grid.keys(), v))
        for v in itertools.product(*grid.values())
    ]

    for params in param_combinations:

        # Create unique run name
        run_name = f"{model_name}_tune_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

        with exp.start_run(run_name):

            # Log hyperparameters
            exp.log_params(params)

            # Build new pipeline with updated params
            tuned_pipeline = Pipeline([
                ("preprocess", preprocessor),
                ("model", pipeline.named_steps["model"].__class__(**params))
            ])

            # Train
            tuned_pipeline.fit(X_train, y_train)

            # Predict
            y_pred = tuned_pipeline.predict(X_test)

            # Metrics
            mae = mean_absolute_error(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)

            # Log metrics
            exp.log_metric("MAE", mae)
            exp.log_metric("RMSE", rmse)
            exp.log_metric("R2", r2)

            # Store results
            tuning_results.append({
                "model": model_name,
                "params": params,
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2,
                "run_name": run_name
            })

            st.write(f"Parameters: {params}")
            st.write(f"➡️ MAE: `{mae:.4f}`, RMSE: `{rmse:.4f}`, R²: `{r2:.4f}`")

st.success("✅ Hyperparameter tuning completed for ALL models!")

# TODO regenerate above cell and ask AI to help wth interpreting the results

In [ ]:
# ============================================
# Section 4 — Hyperparameter Tuning Results Summary
# ============================================

import pandas as pd
import streamlit as st

st.subheader("📊 Hyperparameter Tuning Summary")

# Convert results list into DataFrame
results_df = pd.DataFrame(tuning_results)

# Display full results table with highlighted best metrics
st.write("### 🔍 All Tuning Results")

st.dataframe(
    results_df.style.highlight_max(
        subset=["R2", "MAE", "RMSE"],
        color="lightgreen"
    )
)

# ---------------------------------------------------------
# Identify the best model (highest R²)
# ---------------------------------------------------------
best_idx = results_df["R2"].idxmax()
best_model = results_df.loc[best_idx]

st.subheader("🏆 Best Model Configuration")

st.write(f"**Model:** `{best_model['model']}`")
st.write(f"**R² Score:** `{best_model['R2']:.4f}`")
st.write(f"**MAE:** `{best_model['MAE']:.4f}`")
st.write(f"**RMSE:** `{best_model['RMSE']:.4f}`")

st.write("### 🔧 Best Hyperparameters")
for key, value in best_model["params"].items():
    st.write(f"- **{key}**: `{value}`")

st.write(f"### 🔗 Experiment Run")
st.write(f"Run Name: `{best_model['run_name']}`")

# note: I am getting issues with this seciont of the project. 

I need to come back and try and fix them later. 

# Section 4 - Part 5: Model Registry

In [ ]:
# Now we register the best model to Snowflake's Model 
# Registry for deployment and management 

# Train the final RandomForestRegressor using the best hyperparameters
# ============================================
# Section 4 — Model Registry
# Train Final Best Model for Deployment
# ============================================

import streamlit as st
from sklearn.pipeline import Pipeline

st.subheader("📦 Registering Best Model for Deployment")

# ---------------------------------------------------------
# 1. Extract best model name + hyperparameters
# ---------------------------------------------------------
best_model_name = best_model["model"]
best_params = best_model["params"]

st.write(f"🏆 Best Model Identified: `{best_model_name}`")
st.write("🔧 Best Hyperparameters:")
st.write(best_params)

# ---------------------------------------------------------
# 2. Rebuild the final model pipeline using best hyperparameters
# ---------------------------------------------------------
# Get the class of the model from the original pipeline dictionary
model_class = models[best_model_name].named_steps["model"].__class__

# Build final pipeline
final_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model_class(**best_params))
])

# ---------------------------------------------------------
# 3. Train final model on full training data
# ---------------------------------------------------------
final_pipeline.fit(X_train, y_train)

st.success("✅ Final model trained using best hyperparameters and preprocessing pipeline.") 

In [ ]:
best_params # Displays the best parameters object.

In [ ]:
# Register the best-performing regression model in Snowflake Model Registry
# This cell:
# Registers the model
# logs regression metrics
# Stores sample input 
# creates a versioned model entry

import streamlit as st
from snowflake.ml.registry import Registry
from datetime import datetime
import warnings


st.subheader("📦 Registering Final Model in Snowflake Model Registry")

# ---------------------------------------------------------
# 1. Connect to the Model Registry
# ---------------------------------------------------------
registry = Registry(sesh)

# ---------------------------------------------------------
# 2. Prepare sample input (raw features, not scaled)
# ---------------------------------------------------------
sample_input = X_train.head(5).copy()
sample_input.columns = sample_input.columns.str.replace(" ", "_")

# ---------------------------------------------------------
# 3. Define model name + version
# ---------------------------------------------------------
model_name = "HOUSE_PRICE_MODELER"
model_version = f"v_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# ---------------------------------------------------------
# 4. Register the model (full pipeline)
# ---------------------------------------------------------
warnings.filterwarnings("ignore", category=RuntimeWarning)


model_ref = registry.log_model(
    model=final_pipeline,                     # full pipeline (preprocess + model)
    model_name=model_name,
    version_name=model_version,
    sample_input_data=sample_input,           # stored for inference schema
    metrics={
        "MAE": float(best_model["MAE"]),
        "RMSE": float(best_model["RMSE"]),
        "R2": float(best_model["R2"])
    },
    options={"case_sensitive": True},
    comment=f"Best performing model ({best_model['model']}) from hyperparameter tuning"
)

warnings.resetwarnings()

# ---------------------------------------------------------
# 5. Streamlit confirmation
# ---------------------------------------------------------
st.success("✅ Final model successfully registered in Snowflake Model Registry!")
st.write(f"**Model Name:** `{model_name}`")
st.write(f"**Version:** `{model_version}`")


In [ ]:
// Below shows the available versions in a model
SHOW VERSIONS IN MODEL HOUSE_PRICE_MODELER;

In [ ]:
// Below shows the available functions in a model
SHOW FUNCTIONS IN MODEL HOUSE_PRICE_MODELER

In [ ]:
# Deploy the regression model as a Snowflake service

# What is a snowflake service? 
# A fully managed, scalable endpoint inside Snowflake that hosts your ML model so it can be called by applications, APIs, dashboards, or Streamlit apps.
# Above defintion taken from Microsoftr Copilot.

# This cell: 
# Drops old services 
# Deploys the regression model 
# makes it callable from Streeamlit or APIS


import streamlit as st

st.subheader("🚀 Deploying Model as a Snowflake Service")

# ---------------------------------------------------------
# 1. Drop old service if it exists
# ---------------------------------------------------------
sesh.sql("DROP SERVICE IF EXISTS HOUSE_PRICE_MODELER").collect()

# ---------------------------------------------------------
# 2. Deploy the new service using the registered model
# ---------------------------------------------------------
model_ref.create_service(
    service_name="HOUSE_PRICE_MODELER",
    service_compute_pool="system_compute_pool_cpu",   # CPU pool for regression
    ingress_enabled=True,                             # Allow external access
    gpu_requests=None                                 # No GPU needed for regression
)

st.success("✅ House Price Regression model service created successfully!")
st.write("Your model is now deployed as a scalable Snowflake service and can be called from:")
st.write("- Streamlit apps")
st.write("- Dashboards")
st.write("- External APIs")
st.write("- SQL queries inside Snowflake")

In [ ]:
// Show service endpoints exposed by the model

SHOW SERVICES;

In [ ]:
SHOW ENDPOINTS IN SERVICE HOUSE_PRICE_MODELER;

In [ ]:
SHOW TABLES;

# NOTE BELOW 2 CELLS WONT WORK --> NEED TO BE FIXED.


# Ask AI how these 2 can be changed to the curretn project.

In [ ]:
-- With the model deployed as a setvice, wre can use it to make predictions on new data
-- We do this by:
-- Query the HOUSE_TEST_DATA table
-- Pass features to the model

/* 
HOUSE_PRICE_MODELER 
    The name of the deployed model 

! PREDICT() 
    Snowflake syntax -- Says to call the PREDICT function on that model.

Read like "Use HOUSE_PRICE_MODELER to predict given these features."
*/

SELECT
    HOUSE_PRICE_MODELER!PREDICT(
        OBJECT_CONSTRUCT(
            'SQFT', SQFT,
            'BEDROOMS', BEDROOMS,
            'BATHROOMS', BATHROOMS,
            'HOUSE_TYPE', HOUSE_TYPE,
            'LOT_SIZE', LOT_SIZE,
            'LUXURY', LUXURY
        )
    ) AS predicted_price
FROM SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_TEST_DATA
LIMIT 5;

In [ ]:
SELECT
    AVG("SQFT") AS avg_sqft,
    AVG("BEDROOMS") AS avg_bedrooms,
    AVG("BATHROOMS") AS avg_bathrooms,

    -- Category proportions for LOT_SIZE
    AVG(CASE WHEN "LOT_SIZE" = 'small' THEN 1 ELSE 0 END) AS lot_small,
    AVG(CASE WHEN "LOT_SIZE" = 'medium' THEN 1 ELSE 0 END) AS lot_medium,
    AVG(CASE WHEN "LOT_SIZE" = 'large' THEN 1 ELSE 0 END) AS lot_large,

    -- Category proportions for LUXURY
    AVG(CASE WHEN "LUXURY" = 'basic' THEN 1 ELSE 0 END) AS luxury_basic,
    AVG(CASE WHEN "LUXURY" = 'mid' THEN 1 ELSE 0 END) AS luxury_mid,
    AVG(CASE WHEN "LUXURY" = 'highend' THEN 1 ELSE 0 END) AS luxury_highend,

    -- Category proportions for HOUSE_TYPE
    AVG(CASE WHEN "HOUSE_TYPE" = 'modern' THEN 1 ELSE 0 END) AS type_modern,
    AVG(CASE WHEN "HOUSE_TYPE" = 'ranch' THEN 1 ELSE 0 END) AS type_ranch,
    AVG(CASE WHEN "HOUSE_TYPE" = 'two_story_trad' THEN 1 ELSE 0 END) AS type_two_story_trad,
    AVG(CASE WHEN "HOUSE_TYPE" = 'bungalow' THEN 1 ELSE 0 END) AS type_bungalow,
    AVG(CASE WHEN "HOUSE_TYPE" = 'prairie_suburban' THEN 1 ELSE 0 END) AS type_prairie_suburban,
    AVG(CASE WHEN "HOUSE_TYPE" = 'farmhouse' THEN 1 ELSE 0 END) AS type_farmhouse,
    AVG(CASE WHEN "HOUSE_TYPE" = 'chalet' THEN 1 ELSE 0 END) AS type_chalet,
    AVG(CASE WHEN "HOUSE_TYPE" = 'four_square' THEN 1 ELSE 0 END) AS type_four_square,
    AVG(CASE WHEN "HOUSE_TYPE" = 'skinny' THEN 1 ELSE 0 END) AS type_skinny,
    AVG(CASE WHEN "HOUSE_TYPE" = 'mediterranean' THEN 1 ELSE 0 END) AS type_mediterranean

FROM SNOWFLAKE_LEARNING_DB.PUBLIC.HOUSE_TEST_DATA;